In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import requests
import numpy as np
from collections import Counter
import re

In [15]:
# Pobieramy dane
url = "https://www.gutenberg.org/files/100/100-0.txt"
text = requests.get(url).text
start = text.find("THE SONNETS")
end = text.find("THE END")
text = text[start:end]
text = re.sub(r'\b\d+\b', '', text)

In [16]:
# Tokenizacja na słowa
words = text.split()
word_counts = Counter(words)
vocab = sorted(word_counts, key=word_counts.get, reverse=True)

In [17]:
# Tworzymy słowniki
word_to_index = {word: i for i, word in enumerate(vocab)}
index_to_word = {i: word for i, word in enumerate(vocab)}

In [18]:
# Zamiana tekstu na indeksy
encoded_text = [word_to_index[word] for word in words if word in word_to_index]

In [19]:
# Tworzymy sekwencje do uczenia
seq_length = 10
input_sequences = []
output_words = []

for i in range(len(encoded_text) - seq_length):
    input_sequences.append(encoded_text[i:i+seq_length])
    output_words.append(encoded_text[i+seq_length])

In [20]:
# Zamiana na tensory
input_sequences = torch.tensor(input_sequences, dtype=torch.long)
output_words = torch.tensor(output_words, dtype=torch.long)

In [21]:
# Definicja modelu
class ShakespeareLNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers):
        super(ShakespeareLNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        output, hidden = self.lstm(x, hidden)
        x = self.fc(output[:, -1, :])  # Predykcja kolejnego słowa
        return x, hidden

In [22]:
# Parametry modelu
embedding_dim = 128
hidden_dim = 256
num_layers = 2
vocab_size = len(vocab)

model = ShakespeareLNN(vocab_size, embedding_dim, hidden_dim, num_layers)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)

In [23]:
# Trening
batch_size = 64
dataset = torch.utils.data.TensorDataset(input_sequences, output_words)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

num_epochs = 18  # Trenujemy przez 18 epok

for epoch in range(num_epochs):
    total_loss = 0
    correct_predictions = 0
    total_samples = 0

    for inputs, targets in dataloader:
        optimizer.zero_grad()
        outputs, _ = model(inputs)

        loss = loss_fn(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Obliczanie accuracy
        predicted = torch.argmax(outputs, dim=1)  # Najbardziej prawdopodobne słowo
        correct_predictions += (predicted == targets).sum().item()
        total_samples += targets.size(0)

    # Perplexity = e^(średnia strata)
    avg_loss = total_loss / len(dataloader)
    perplexity = np.exp(avg_loss)
    accuracy = correct_predictions / total_samples * 100

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}, Perplexity: {perplexity:.4f}, Accuracy: {accuracy:.2f}%")

Epoch 1/18, Loss: 7.3513, Perplexity: 1558.1892, Accuracy: 2.19%
Epoch 2/18, Loss: 6.9154, Perplexity: 1007.6794, Accuracy: 2.84%
Epoch 3/18, Loss: 6.6446, Perplexity: 768.6312, Accuracy: 4.08%
Epoch 4/18, Loss: 6.2291, Perplexity: 507.3091, Accuracy: 5.74%
Epoch 5/18, Loss: 5.6590, Perplexity: 286.8681, Accuracy: 7.74%
Epoch 6/18, Loss: 4.9271, Perplexity: 137.9836, Accuracy: 11.27%
Epoch 7/18, Loss: 4.1039, Perplexity: 60.5773, Accuracy: 19.01%
Epoch 8/18, Loss: 3.2538, Perplexity: 25.8873, Accuracy: 32.76%
Epoch 9/18, Loss: 2.4634, Perplexity: 11.7443, Accuracy: 49.55%
Epoch 10/18, Loss: 1.8388, Perplexity: 6.2891, Accuracy: 62.60%
Epoch 11/18, Loss: 1.3568, Perplexity: 3.8837, Accuracy: 73.33%
Epoch 12/18, Loss: 1.0106, Perplexity: 2.7474, Accuracy: 81.05%
Epoch 13/18, Loss: 0.7450, Perplexity: 2.1065, Accuracy: 86.96%
Epoch 14/18, Loss: 0.5685, Perplexity: 1.7656, Accuracy: 90.68%
Epoch 15/18, Loss: 0.4285, Perplexity: 1.5350, Accuracy: 93.76%
Epoch 16/18, Loss: 0.3480, Perplexity

In [25]:
# Zapisujemy model
torch.save(model.state_dict(), "model.pth")
print("Model zapisany jako model.pth")

Model zapisany jako model.pth


In [26]:
import random
# Funkcja generowania tekstu z temperature scaling
def generate_text(model, start_word, length=50, temperature=1.0):
    model.eval()
    words_generated = [start_word]

    word_index = word_to_index.get(start_word, random.randint(0, len(vocab) - 1))
    input_seq = torch.tensor([[word_index]], dtype=torch.long)

    hidden = None

    for _ in range(length):
        with torch.no_grad():
            output, hidden = model(input_seq, hidden)

            # Sampling z temperaturą
            probabilities = torch.softmax(output / temperature, dim=-1)
            next_word_index = torch.multinomial(probabilities, num_samples=1).item()

            words_generated.append(index_to_word[next_word_index])
            input_seq = torch.tensor([[next_word_index]], dtype=torch.long)

    return " ".join(words_generated)



In [27]:
# Przykładowe generowanie tekstu
print(generate_text(model, start_word="Shall", length=50, temperature=0.8))

Shall I keep forth, Her first my self, hath ’scaped this sorrow, They in the rearward of a conquered woe, Give not a windy night a rainy morrow, To linger out a purposed overthrow. If thou goest one, on you much left more happy time all all my loves, you be


In [28]:
pip install gradio

In [29]:
import gradio as gr

# Załaduj model
model.load_state_dict(torch.load("model.pth"))
model.eval()

# Funkcja generująca tekst
def generate_text_gradio(start_word, length=50, temperature=1.0):
    return generate_text(model, start_word, length, temperature)

# Tworzymy interfejs
iface = gr.Interface(
    fn=generate_text_gradio,  # Funkcja, którą wywołujemy
    inputs=[
        gr.Textbox(label="Starting Word", value="Shall"),  # Pole wejściowe na słowo początkowe
        gr.Slider(minimum=10, maximum=100, step=1, label="Text Length", value=50),  # Suwak do długości tekstu
        gr.Slider(minimum=0.1, maximum=2.0, step=0.1, label="Temperature", value=1.0)  # Suwak do temperatury
    ],
    outputs=gr.Textbox(label="Generated Text")  # Wyjściowe pole tekstowe z wygenerowanym tekstem
)

# Uruchamiamy aplikację
iface.launch()


<ipython-input-29-cde3254ebc10>:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("model.pth"))


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5055ed755118b3b356.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
